In [ ]:
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import json

In [ ]:
def calibrate_camera_from_checkerboard(
    images_glob: str,
    pattern_size=(9, 6),        # inner corners (cols, rows)
    square_size_m=0.025,         # 3 cm squares -> set to your printed square size
    visualize=False,
    K=None
):
    """
    Calibrates a pinhole camera + distortion using a checkerboard.

    Args:
      images_glob: e.g. "/path/to/calib/*.JPG"
      pattern_size: (nx, ny) number of INNER corners
      square_size_m: physical size of one square edge (meters). Only affects translation scale,
                     NOT fx/fy in pixels.
      visualize: show detected corners for debugging

    Returns:
      K (3x3), dist (1x5 or 1x8), rvecs, tvecs, rms_error
    """
    # Prepare object points: (0,0,0), (1,0,0), ..., (nx-1, ny-1, 0) scaled by square_size
    nx, ny = pattern_size
    objp = np.zeros((nx * ny, 3), np.float32)
    objp[:, :2] = np.mgrid[0:nx, 0:ny].T.reshape(-1, 2)
    objp *= float(square_size_m)

    objpoints = []  # 3D points in world coordinate
    imgpoints = []  # 2D points in image plane

    images = sorted(glob.glob(images_glob))
    if not images:
        raise FileNotFoundError(f"No images found for glob: {images_glob}")

    print(f"Total number of available images = {len(images)}")

    # Corner refinement criteria
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 1e-6)

    image_size = None

    for fname in images:
        img = cv2.imread(fname)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        if image_size is None:
            image_size = (gray.shape[1], gray.shape[0])  # (w, h)
            print(f"image size = {image_size}")

        # Find checkerboard corners
        found, corners = cv2.findChessboardCorners(
            gray, (nx, ny),
            flags=cv2.CALIB_CB_ADAPTIVE_THRESH
                  + cv2.CALIB_CB_NORMALIZE_IMAGE
                #   + cv2.CALIB_CB_FAST_CHECK
                  + cv2.CALIB_CB_ACCURACY
        )

        if not found:
            print(f"[WARN] No corners found: {fname}")
            continue

        # Refine corners to subpixel accuracy
        corners_refined = cv2.cornerSubPix(
            gray, corners, winSize=(11, 11), zeroZone=(-1, -1), criteria=criteria
        )

        objpoints.append(objp)
        imgpoints.append(corners_refined)

        if visualize:
            vis = img.copy()
            cv2.drawChessboardCorners(vis, (nx, ny), corners_refined, found)
            vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
            plt.imshow(vis_rgb)
            plt.show()

    if len(objpoints) < 10:
        raise RuntimeError(f"Too few valid images ({len(objpoints)}). Aim for 15–30+.")
    
    print(f"Images with corners = {len(objpoints)}")

    cal_flags = cv2.CALIB_FIX_K3
    if K is not None:
        cal_flags += cv2.CALIB_USE_INTRINSIC_GUESS

    # Calibrate
    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, image_size, K, None, flags=cal_flags
    )

    return image_size[0], image_size[1],  K, dist, rvecs, tvecs, rms

In [ ]:
K_from_metadata = np.array([[2687.996, 0, 2016],
                            [0, 2687.996, 1512],
                            [0, 0, 1]])

In [ ]:
calibration_images = "G:/Mary/Picture/drone/calibration/4_3/images/*.JPG"

H, W, K, dist, rvecs, tvecs, rms = calibrate_camera_from_checkerboard(
        images_glob=calibration_images,
        pattern_size=(9, 6),
        square_size_m=0.025,  # set to your printed square size
        visualize=True,
        K = None
    )

print("RMS reprojection error:", rms)
print("K (intrinsics):\n", K)
print("dist (distortion):\n", dist)

In [ ]:
fx = K[0, 0]
cx = K[0, 2]
fy = K[1, 1]
cy = K[1, 2]

print(f"fx = {fx}, fy = {fy}")
print(f"cx = {cx}, cy = {cy}")

In [ ]:
k1 = dist[0, 0]
k2 = dist[0, 1]
k3 = dist[0, 4]
p1 = dist[0, 2]
p2 = dist[0, 3]

print(f"k1 = {k1}, k2 = {k2}, k3 = {k3}")
print(f"p1 = {p1}, p2 = {p2}")

In [ ]:
cam_calib = {"camera_type":"OPENCV", "H":H, "W":W, "fx":fx, "fy":fy, "cx":cx, "cy":cy, "k1":k1, "k2":k2, "k3":k3, "p1":p1, "p2":p2, "RMS":rms}

In [ ]:
cal_file = "G:/Mary/Picture/drone/calibration/cal_val_4_3_w_k.json"

with open(cal_file, 'w') as f:
    json.dump(cam_calib, f, indent=4)